# 06 · Temporal Fusion Transformer (PyTorch Forecasting)
A deep, attention-based model that uses static info (store, item, category), known-future inputs (price, SNAP,
calendar, events) and the past sales history, and outputs **quantiles** directly.

**Runtime:** TFT is slow on CPU. Defaults below (60 series, 5 epochs) are a smoke-test; on a Colab/Kaggle GPU raise
`TFT_SERIES` (e.g. 300–600) and `TFT_EPOCHS` (20–30) via environment variables, then copy
`data/outputs/pred_tft.parquet` and `metrics_tft.csv` back into the repo.

In [ ]:
import sys; sys.path.insert(0, "..")
import os, warnings, logging; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json
from src.config import *
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": .3})

import torch, lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
from src.metrics import wape, rmsse
pl.seed_everything(SEED)
N_SERIES = int(os.environ.get("TFT_SERIES", 60)); MAX_EPOCHS = int(os.environ.get("TFT_EPOCHS", 5))
print(f"series={N_SERIES}  epochs={MAX_EPOCHS}  device={'cuda' if torch.cuda.is_available() else 'cpu'}")

cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id", "date", "sales", "sell_price", "snap",
        "price_rel", "promo_proxy", "days_to_event", "days_since_event", "dow", "month", "event_type_1"]
feat = pd.read_parquet(PROC_DIR / "features.parquet", columns=cols)
test_start = feat.date.max() - pd.Timedelta(days=HORIZON - 1)
top = feat[feat.date < test_start].groupby("id")["sales"].sum().nlargest(N_SERIES).index
d = feat[feat.id.isin(top)].copy()
d["time_idx"] = (d.date - d.date.min()).dt.days
for c in ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id", "dow", "month", "event_type_1"]:
    d[c] = d[c].astype(str)
d["sales"] = d["sales"].astype(float)
test_idx0 = int((test_start - d.date.min()).days)

## Datasets: train → validation (28 days before test, for early stopping) → test

In [ ]:
training = TimeSeriesDataSet(
    d[d.time_idx < test_idx0 - HORIZON], time_idx="time_idx", target="sales", group_ids=["id"],
    min_encoder_length=28, max_encoder_length=56, min_prediction_length=1, max_prediction_length=HORIZON,
    static_categoricals=["store_id", "item_id", "dept_id", "cat_id", "state_id"],
    time_varying_known_categoricals=["dow", "month", "event_type_1"],
    time_varying_known_reals=["time_idx", "sell_price", "snap", "price_rel", "promo_proxy", "days_to_event", "days_since_event"],
    time_varying_unknown_reals=["sales"],
    target_normalizer=GroupNormalizer(groups=["id"], transformation="softplus"),
    add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True, allow_missing_timesteps=True)
validation = TimeSeriesDataSet.from_dataset(training, d[d.time_idx < test_idx0], predict=True, stop_randomization=True)
testing = TimeSeriesDataSet.from_dataset(training, d, predict=True, stop_randomization=True)
train_dl = training.to_dataloader(train=True, batch_size=128, num_workers=0)
val_dl = validation.to_dataloader(train=False, batch_size=256, num_workers=0)
test_dl = testing.to_dataloader(train=False, batch_size=256, num_workers=0)

In [ ]:
tft = TemporalFusionTransformer.from_dataset(
    training, learning_rate=0.03, hidden_size=16, attention_head_size=2, dropout=0.1, hidden_continuous_size=8,
    loss=QuantileLoss(quantiles=QUANTILES), output_size=len(QUANTILES), reduce_on_plateau_patience=2)
print("parameters:", sum(p.numel() for p in tft.parameters()))
trainer = pl.Trainer(max_epochs=MAX_EPOCHS, accelerator="auto", gradient_clip_val=0.1, logger=False,
                     enable_progress_bar=False, enable_checkpointing=False,
                     callbacks=[EarlyStopping(monitor="val_loss", patience=3, mode="min")])
trainer.fit(tft, train_dataloaders=train_dl, val_dataloaders=val_dl)

## Predict the test window (quantiles) and score against LightGBM on the same series

In [ ]:
pred = tft.predict(test_dl, mode="quantiles", return_index=True)
out, idx = (pred.output, pred.index) if hasattr(pred, "output") else (pred[0], pred[1])
out = out.cpu().numpy() if hasattr(out, "cpu") else np.asarray(out)       # (n_series, 28, n_quantiles)
qcols = [f"q{int(q*100):02d}" for q in QUANTILES]
rows = []
for i, (_, r) in enumerate(idx.iterrows()):
    sid = r["id"]; dates = d.loc[d.id == sid].set_index("time_idx").loc[int(r["time_idx"]): int(r["time_idx"]) + HORIZON - 1, "date"].values
    rows.append(pd.DataFrame(np.clip(np.sort(out[i], axis=1), 0, None), columns=[f"tft_{c}" for c in qcols]).assign(id=sid, date=dates))
tft_pred = pd.concat(rows, ignore_index=True)
tft_pred["pred_tft"] = tft_pred["tft_q50"]
pt = pd.read_parquet(OUT_DIR / "pred_point.parquet")
cmp = pt[["id", "date", "sales", "pred_lgbm", "pred_snaive"]].merge(tft_pred, on=["id", "date"])
hist = feat[feat.date < test_start][["id", "date", "sales"]]
res = pd.DataFrame([dict(model=m.replace("pred_", ""), scope=f"TFT subset ({cmp.id.nunique()} series)", n_series=cmp.id.nunique(),
        WAPE=wape(cmp.sales, cmp[m]), RMSSE=rmsse(hist[hist.id.isin(cmp.id)], cmp, m)) for m in ["pred_snaive", "pred_lgbm", "pred_tft"]]).round(4)
tft_pred.to_parquet(OUT_DIR / "pred_tft.parquet"); res.to_csv(OUT_DIR / "metrics_tft.csv", index=False); res

## Interpretability: what does the TFT attend to?

In [ ]:
raw, x = tft.predict(val_dl, mode="raw", return_x=True)[:2] if False else (None, None)
interp = tft.interpret_output(tft.predict(val_dl, mode="raw", return_x=False), reduction="sum")
tft.plot_interpretation(interp); plt.savefig(FIG_DIR / "06_tft_interpretation.png", dpi=110, bbox_inches="tight"); plt.show()

**Honest comparison:** on M5-style data a well-tuned LightGBM is often as good as TFT (the winning M5 solutions
were tree models). Report both, and say what TFT adds: native quantiles, interpretability and attention weights.